# Module 4: FFT -- What Frequencies Are Hiding in Your Signal?

This tutorial walks through the concepts step by step. Each section includes an explanation, an example, and an exercise.

## Step 1: Setup

Import libraries for FFT analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Ready!')

**Exercise 1:** Try it yourself!

## Step 2: The Core Idea: Time Domain vs Frequency Domain

**Clinical scenario:** A neurologist reviewing an EEG says: "I see a lot of theta activity." How does she know? She's not counting cycles by hand -- the machine computed a **power spectrum** using the Fourier Transform.

The key insight: **any signal is just a sum of sine waves at different frequencies.** The FFT decomposes a signal into those individual components.

- **Time domain**: voltage changing over time (the wiggly line)
- **Frequency domain**: which frequencies are present and how strong each one is (the "ingredients list")

The FFT converts from the first view to the second.

**Exercise 2:** Try it yourself!

## Step 3: Build a Multi-Component Signal and Compute FFT

Let's build a signal from known components and verify that the FFT recovers them.

In [ ]:
# Signal parameters
fs = 512  # Sampling rate
duration = 2.0
t = np.arange(0, duration, 1/fs)
N = len(t)

# Build a signal from three known components
components = [
    (3, 1.0, '#2c3e50', 'Delta (3 Hz)'),
    (10, 1.5, '#8e44ad', 'Alpha (10 Hz)'),
    (25, 0.8, '#e67e22', 'Beta (25 Hz)'),
]

# Sum of sine waves
composite = np.zeros_like(t)
for freq, amp, _, _ in components:
    composite += amp * np.sin(2 * np.pi * freq * t)

# Compute FFT
yf = np.abs(fft(composite))[:N//2] * 2 / N  # Single-sided amplitude spectrum
freqs = fftfreq(N, 1/fs)[:N//2]

# Plot both domains
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Time domain
axes[0].plot(t, composite, color='#2c3e50', linewidth=1)
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Time Domain -- looks complex', fontsize=12)
axes[0].set_xlim(0, 1)

# Frequency domain
axes[1].plot(freqs, yf, color='#8e44ad', linewidth=1.5)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')
axes[1].set_title('Frequency Domain (FFT) -- clear peaks!', fontsize=12)
axes[1].set_xlim(0, 50)

# Mark the known components
for freq, amp, color, label in components:
    axes[1].axvline(freq, color=color, linestyle='--', alpha=0.7)
    axes[1].annotate(label, xy=(freq, amp), xytext=(freq+2, amp),
                     fontsize=9, color=color)

plt.suptitle('Same Signal, Two Views', fontsize=13)
plt.tight_layout()
plt.show()

print('The FFT perfectly recovers the three frequencies we put in.')
print('The messy-looking time domain signal has a clean, simple spectrum.')

**Exercise 3:** Try it yourself!

## Step 4: Mystery Signal: Identify the Hidden Frequencies

Here's a composite signal. It looks messy in the time domain. Can you guess what frequencies are hiding inside before running the FFT?

Try three clinical scenarios:
1. Normal awake EEG
2. Deep sleep
3. A noisy recording

In [ ]:
np.random.seed(42)

mysteries = {
    'Normal Awake EEG': {
        'components': [(10, 1.2), (18, 0.6), (5, 0.4)],
        'noise': 0.3,
        'answer': 'Dominant alpha (10 Hz), some beta (18 Hz), touch of theta (5 Hz)'
    },
    'Deep Sleep': {
        'components': [(2, 2.5), (6, 0.8)],
        'noise': 0.2,
        'answer': 'Massive delta (2 Hz) with some theta (6 Hz) -- no alpha/beta'
    },
    'Noisy Recording': {
        'components': [(10, 1.0), (60, 1.5)],
        'noise': 0.4,
        'answer': 'Alpha (10 Hz) + 60 Hz power line noise -- needs a notch filter!'
    },
}

fig, axes = plt.subplots(3, 2, figsize=(12, 9))

for idx, (name, info) in enumerate(mysteries.items()):
    # Build signal
    y = np.zeros(N)
    for freq, amp in info['components']:
        y += amp * np.sin(2 * np.pi * freq * t)
    y += info['noise'] * np.random.randn(N)
    
    # Time domain
    axes[idx, 0].plot(t, y, color='#2c3e50', linewidth=0.8)
    axes[idx, 0].set_title(f'{name} (time domain)', fontsize=11)
    axes[idx, 0].set_xlim(0, 1)
    axes[idx, 0].set_ylabel('Amplitude')
    
    # FFT
    yf = np.abs(fft(y))[:N//2] * 2 / N
    axes[idx, 1].plot(freqs, yf, color='#8e44ad', linewidth=1.5)
    axes[idx, 1].set_title(f'{name} (FFT spectrum)', fontsize=11)
    axes[idx, 1].set_xlim(0, 70)
    axes[idx, 1].set_ylabel('Magnitude')
    
    # Print answer
    print(f'{name}: {info["answer"]}')

axes[-1, 0].set_xlabel('Time (seconds)')
axes[-1, 1].set_xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

**Exercise 4:** Try it yourself!

## Step 5: Frequency Resolution: Why Recording Duration Matters

Can the FFT tell apart two close frequencies, like 9 Hz and 11 Hz? It depends on **how long you record**.

The frequency resolution of the FFT is:

$$\Delta f = \frac{1}{T}$$

where $T$ is the recording duration in seconds.
- 1-second recording: can resolve frequencies 1 Hz apart
- 0.5-second recording: only 2 Hz apart
- 2-second recording: 0.5 Hz apart

**Clinical implication:** A 1-second EEG epoch can tell alpha from beta, but to precisely characterize 9 Hz vs 11 Hz, you need several seconds of clean data.

In [ ]:
durations = [0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

f1, f2 = 9, 11  # Two close frequencies
a1, a2 = 1.0, 0.8

for ax, dur in zip(axes.flat, durations):
    # Generate signal
    t_d = np.arange(0, dur, 1/fs)
    N_d = len(t_d)
    y_d = a1 * np.sin(2*np.pi*f1*t_d) + a2 * np.sin(2*np.pi*f2*t_d)
    
    # FFT
    yf_d = np.abs(fft(y_d))[:N_d//2] * 2 / N_d
    freqs_d = fftfreq(N_d, 1/fs)[:N_d//2]
    
    # Plot spectrum (zoom into 5-15 Hz)
    mask = freqs_d <= 20
    ax.plot(freqs_d[mask], yf_d[mask], color='#8e44ad', linewidth=1.5)
    ax.fill_between(freqs_d[mask], yf_d[mask], alpha=0.15, color='#8e44ad')
    ax.axvline(9, color='#e74c3c', linestyle='--', alpha=0.5)
    ax.axvline(11, color='#2980b9', linestyle='--', alpha=0.5)
    
    delta_f = 1 / dur
    resolved = 'RESOLVED' if delta_f <= 2 else 'MERGED'
    color = '#27ae60' if delta_f <= 2 else '#c0392b'
    ax.set_title(f'T={dur:.1f}s  |  delta_f={delta_f:.2f} Hz  |  {resolved}',
                 fontsize=10, color=color)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude')

plt.suptitle('Frequency Resolution: 9 Hz + 11 Hz -- Can the FFT Tell Them Apart?',
             fontsize=13)
plt.tight_layout()
plt.show()

print('At 0.5s, the two peaks are merged into one blob.')
print('At 1.0s, they start to separate.')
print('At 2.0s and above, they are clearly distinct.')
print('More data = more precision.')

**Exercise 5:** Try it yourself!

## Step 6: How Sampling Rate Affects FFT

Two things matter for the FFT:

1. **Sampling rate** ($f_s$) determines the maximum frequency you can see: $f_{max} = f_s / 2$ (Nyquist)
2. **Recording duration** ($T$) determines how finely you can resolve frequencies: $\Delta f = 1/T$

Let's see this interplay.

In [ ]:
# Same signal, different sampling rates
signal_freqs = [5, 15, 45]  # Hz
dur_demo = 2.0

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, fs_demo in zip(axes, [64, 128, 512]):
    t_d = np.arange(0, dur_demo, 1/fs_demo)
    N_d = len(t_d)
    
    # Build signal with all three frequencies
    y_d = sum(np.sin(2*np.pi*f*t_d) for f in signal_freqs)
    
    # FFT
    yf_d = np.abs(fft(y_d))[:N_d//2] * 2 / N_d
    freqs_d = fftfreq(N_d, 1/fs_demo)[:N_d//2]
    
    ax.plot(freqs_d, yf_d, color='#2980b9', linewidth=1.5)
    ax.fill_between(freqs_d, yf_d, alpha=0.1, color='#2980b9')
    
    # Mark expected frequencies
    for f in signal_freqs:
        if f <= fs_demo / 2:
            ax.axvline(f, color='#27ae60', linestyle='--', alpha=0.5)
        else:
            ax.axvline(f, color='#c0392b', linestyle=':', alpha=0.3)
            ax.text(f, 0.05, f'{f} Hz\n(aliased!)', fontsize=8, color='#c0392b',
                    ha='center')
    
    nyquist = fs_demo / 2
    ax.set_title(f'fs={fs_demo} Hz  |  Nyquist={nyquist:.0f} Hz', fontsize=11)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude')
    ax.set_xlim(0, max(freqs_d))

plt.suptitle('Same Signal (5+15+45 Hz), Different Sampling Rates', fontsize=13)
plt.tight_layout()
plt.show()

print('At fs=64 Hz: Nyquist=32 Hz. The 45 Hz component is ALIASED.')
print('At fs=128 Hz: Nyquist=64 Hz. All three components are visible.')
print('At fs=512 Hz: Nyquist=256 Hz. All components clearly visible, with room to spare.')

**Exercise 6:** Try it yourself!

## Step 7: Exercise: Find the Hidden Frequencies

Create a signal with 5, 15, and 25 Hz components, compute the FFT, and verify you can find all three.

**Exercise 7:** Try it yourself!

In [ ]:
# YOUR CODE HERE
# Task: Create a signal with 5, 15, and 25 Hz components. Use FFT to find them.
#
# Steps:
# 1. Set up: fs = 256, duration = 2.0, t = np.arange(0, duration, 1/fs)
# 2. Build signal: y = sin(5Hz) + 0.7*sin(15Hz) + 0.4*sin(25Hz)
# 3. Compute FFT:
#    N = len(y)
#    yf = np.abs(fft(y))[:N//2] * 2 / N
#    freqs = fftfreq(N, 1/fs)[:N//2]
# 4. Plot the spectrum. Do you see three peaks?
# 5. Label the peaks with their frequencies.


## Step 8: Exercise: Sampling Rate and FFT

Explore how changing the sampling rate affects what the FFT can show you.

**Exercise 8:** Try it yourself!

In [ ]:
# YOUR CODE HERE
# Task: Take a 50 Hz signal and compute its FFT at three different sampling rates:
# fs = 80 Hz, fs = 128 Hz, and fs = 512 Hz.
#
# For each, plot the FFT spectrum side by side.
# Questions to answer:
# 1. At fs=80 Hz, can you see the 50 Hz component? (Hint: Nyquist = 40 Hz)
# 2. At fs=128 Hz, where does the peak appear?
# 3. At fs=512 Hz, where does the peak appear?
#
# Hint: Use the same plotting approach as the 'How Sampling Rate Affects FFT' section.
